# Multi-Index RAG Pipeline
### 2 PDFs → 2 Vector Stores → Parallel Retrieval → Merge → Answer

In [ ]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load Both PDFs

In [2]:
loader_kalam = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages_kalam = loader_kalam.load()

loader_altman = PyPDFLoader("Sam_Altman.pdf")
pages_altman = loader_altman.load()

print(f"Kalam PDF - Total Pages: {len(pages_kalam)}")
print(f"Altman PDF - Total Pages: {len(pages_altman)}")

Kalam PDF - Total Pages: 30
Altman PDF - Total Pages: 21


## Step 2: Chunk Both PDFs

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

chunks_kalam = text_splitter.split_documents(pages_kalam)
chunks_altman = text_splitter.split_documents(pages_altman)

print(f"Kalam Chunks: {len(chunks_kalam)}")
print(f"Altman Chunks: {len(chunks_altman)}")

Kalam Chunks: 136
Altman Chunks: 90


## Step 3: Create Two Separate Vector Stores (Index A & Index B)

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

# Index A - Abdul Kalam
vector_store_kalam = FAISS.from_documents(chunks_kalam, embeddings)
print(f"Index A (Kalam) - {vector_store_kalam.index.ntotal} vectors")

# Index B - Sam Altman
vector_store_altman = FAISS.from_documents(chunks_altman, embeddings)
print(f"Index B (Altman) - {vector_store_altman.index.ntotal} vectors")

Index A (Kalam) - 136 vectors
Index B (Altman) - 90 vectors


## Step 4: Create Retrievers for Both Indexes

In [5]:
retriever_kalam = vector_store_kalam.as_retriever(search_kwargs={"k": 3})
retriever_altman = vector_store_altman.as_retriever(search_kwargs={"k": 3})

## Step 5: Parallel Retrieval + Merge + Answer

In [6]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

def multi_index_query(query):
    # Parallel retrieval across both indexes
    docs_kalam = retriever_kalam.invoke(query)
    docs_altman = retriever_altman.invoke(query)

    # Merge layer - combine results from both indexes
    all_docs = docs_kalam + docs_altman

    # Deduplicate by content
    seen = set()
    merged_docs = []
    for doc in all_docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            merged_docs.append(doc)

    # Build context
    context = "\n\n".join([doc.page_content for doc in merged_docs])

    # LLM answers using merged context
    prompt = f"""Answer the question based only on the following context:\n\n{context}\n\nQuestion: {query}\nAnswer:"""

    response = llm.invoke(prompt)
    return response.content, merged_docs

## Step 6: Test Queries

In [7]:
# Query about Kalam
query = "What role did Abdul Kalam play in India's missile program?"
answer, docs = multi_index_query(query)

print(f"Query: {query}")
print(f"Answer: {answer}")
print(f"\nRetrieved {len(docs)} chunks")

Query: What role did Abdul Kalam play in India's missile program?
Answer: Abdul Kalam played a major role in the development of India's missile program, including the development of ballistic missiles such as Agni and Prithvi, and launch vehicle technology. He was known as the "Missile Man of India" for his work in this area.

Retrieved 6 chunks


In [8]:
# Query about Sam Altman
query = "What is Sam Altman known for?"
answer, docs = multi_index_query(query)

print(f"Query: {query}")
print(f"Answer: {answer}")
print(f"\nRetrieved {len(docs)} chunks")

Query: What is Sam Altman known for?
Answer: Sam Altman is known for being the CEO of OpenAI, the company behind ChatGPT, a popular AI chatbot.

Retrieved 6 chunks


In [9]:
# Cross-document query - needs both indexes
query = "Compare the career paths of Abdul Kalam and Sam Altman"
answer, docs = multi_index_query(query)

print(f"Query: {query}")
print(f"Answer: {answer}")
print(f"\nRetrieved {len(docs)} chunks")

Query: Compare the career paths of Abdul Kalam and Sam Altman
Answer: Based on the provided context, here's a comparison of the career paths of Abdul Kalam and Sam Altman:

**Abdul Kalam**

* Started as a scientist and engineer, working on various projects, including the development of India's first indigenous satellite, Aryabhata.
* Became the President of India from 2002 to 2007, serving as the 11th President of India.
* After leaving office, he wrote several books, including "A P J Abdul Kalam: The Visionary of India" and "The Kalam Effect: My Years with the President".
* He also appeared in several documentaries and films, including "A Little Dream" and "People's President".
* Currently, he is a public figure and a motivational speaker, using his platform to inspire and promote education and scientific research.

**Sam Altman**

* Started as a entrepreneur and programmer, co-founding several companies, including Reddit and Y Combinator.
* Became the CEO of Reddit in 2015, leading t

## Retrieved Source Chunks

In [10]:
for i, doc in enumerate(docs):
    source = doc.metadata.get('source', 'unknown')
    page = doc.metadata.get('page', 0) + 1
    print(f"\n--- Chunk {i+1} (Source: {source}, Page {page}) ---")
    print(doc.page_content[:300])


--- Chunk 1 (Source: A._P._J._Abdul_Kalam.pdf, Page 16) ---
42. "With him at the helm, there is hope that things might change" (http://www.rediff.com/news/2
002/jul/11spec.htm). Rediff.com. Archived (https://web.archive.org/web/20150729224243/htt
p://www.rediff.com/news/2002/jul/11spec.htm) from the original on 29 July 2015.
43. "SP to support Kalam for Pres

--- Chunk 2 (Source: A._P._J._Abdul_Kalam.pdf, Page 16) ---
Retrieved 1 March 2012.
45. "Narayanan opts out, field clear for Kalam" (http://www.rediff.com/news/2002/jun/11prez5.ht
m). Rediff.com. 11 June 2002. Archived (https://web.archive.org/web/20130518112930/htt
p://www.rediff.com/news/2002/jun/11prez5.htm) from the original on 18 May 2013. Retrieved
1 M

--- Chunk 3 (Source: A._P._J._Abdul_Kalam.pdf, Page 11) ---
A P J Abdul Kalam: The Visionary of India by K Bhushan, G Katyal; A P H Pub Corp,
2002.[176]
The Kalam Effect: My Years with the President by P M Nair; HarperCollins, 2008.[177]
My Days With Mahatma Abdul Kalam by F

Test Questions

1. "What missiles did Abdul Kalam develop?"
2. "What company does Sam Altman lead?"
3. "Compare the educational backgrounds of Kalam and Altman"
4. "Who became president - Kalam or Altman?"
5. "What awards did Abdul Kalam receive?"